In [3]:
##import libraries
import numpy as np
import pandas as pd
import matplotlib as mlt

In [4]:
##load Data sets
file_path = 'Manufacturing_Line_Productivity.xlsx'

df_productivity = pd.read_excel(file_path,sheet_name='Line productivity' ,parse_dates=['Date', 'Start Time', 'End Time'],date_format='%Y-%m-%d %H:%M:%S')
df_products     = pd.read_excel(file_path, sheet_name='Products')
df_factors      = pd.read_excel(file_path, sheet_name='Downtime factors')
df_downtime     = pd.read_excel(file_path, sheet_name='Line downtime', skiprows=1)

In [5]:
df_productivity = pd.read_excel(file_path, sheet_name='Line productivity', parse_dates=['Date', 'Start Time', 'End Time'])
df_productivity.columns = df_productivity.columns.str.strip()
def fix_time_diff(row):
    start_min = row['Start Time'].hour * 60 + row['Start Time'].minute
    end_min = row['End Time'].hour * 60 + row['End Time'].minute
    diff = end_min - start_min
    if diff < 0:
        return diff + 1440
    return diff
df_productivity['Production_duration'] = df_productivity.apply(fix_time_diff, axis=1)
df_productivity['Date'] = pd.to_datetime(df_productivity['Date'], dayfirst=True, errors='coerce')
df_productivity['Start Time'] = df_productivity['Start Time'].dt.time
df_productivity['End Time'] = df_productivity['End Time'].dt.time
df_productivity.rename(columns={'Product': 'Product_Id'}, inplace=True)

def get_shift(start_time):
    hour = start_time.hour
    if 6 <= hour < 14:
        return 'Morning'
    elif 14 <= hour < 22:
        return 'Afternoon'
    else: 
        return 'Night'

df_productivity['shift'] = df_productivity['Start Time'].apply(get_shift)
df_productivity.tail(5)

C:\Users\Midoz\AppData\Local\Temp\ipykernel_10444\2818954475.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_productivity = pd.read_excel(file_path, sheet_name='Line productivity', parse_dates=['Date', 'Start Time', 'End Time'])
C:\Users\Midoz\AppData\Local\Temp\ipykernel_10444\2818954475.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_productivity = pd.read_excel(file_path, sheet_name='Line productivity', parse_dates=['Date', 'Start Time', 'End Time'])


,Date,Product_Id,Batch,Operator,Start Time,End Time,Production_duration,shift
33,2024-09-02,CO-2L,422144,Dennis,12:18:00,14:50:00,152,Morning
34,2024-09-02,CO-2L,422145,Charlie,14:50:00,16:50:00,120,Afternoon
35,2024-09-02,CO-2L,422146,Charlie,16:50:00,19:30:00,160,Afternoon
36,2024-09-02,CO-2L,422147,Charlie,19:30:00,22:55:00,205,Afternoon
37,2024-09-03,CO-2L,422148,Mac,22:55:00,01:05:00,130,Night


In [6]:
df_productivity.info()
print('number of duplicates :' + str(df_productivity.duplicated().sum()))
### cheking coulmns name
##df_productivity.columns
df_productivity.rename(columns={'Product': 'Product_Id'}, inplace=True)
df_productivity.rename(columns={'Batch': 'Batch_Id'}, inplace=True)
df_productivity.rename(columns={'Operator': 'Operator_name'}, inplace=True)
df_productivity= df_productivity.fillna('fill missing value')
##df_productivity.describe()
df_productivity.isnull().sum()
df_productivity.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 38 non-null     datetime64[ns]
 1   Product_Id           38 non-null     object        
 2   Batch                38 non-null     int64         
 3   Operator             38 non-null     object        
 4   Start Time           38 non-null     object        
 5   End Time             38 non-null     object        
 6   Production_duration  38 non-null     int64         
 7   shift                38 non-null     object        
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 2.5+ KB
number of duplicates :0


,Date,Product_Id,Batch_Id,Operator_name,Start Time,End Time,Production_duration,shift
0,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00,135,Morning
1,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00,100,Afternoon
2,2024-08-29,LE-600,422113,Mac,15:45:00,17:35:00,110,Afternoon
3,2024-08-29,LE-600,422114,Mac,17:35:00,19:15:00,100,Afternoon
4,2024-08-29,LE-600,422115,Charlie,19:15:00,20:39:00,84,Afternoon


In [7]:
df_products.info()
# استبدال "2 L" بـ "2000 ml" مباشرة في العمود
## هنا عملنا replace علي طول 

##df_products
### cheking coulmns name
##df_products.columns
df_products.rename(columns={'Product': 'Product_Id'}, inplace=True)
df_products= df_products.fillna('fill missing value')
df_products.isnull().sum()
df_products['Size'] = df_products['Size'].replace('2 L', '2000 ml')
df_products.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Product         6 non-null      object
 1   Flavor          6 non-null      object
 2   Size            6 non-null      object
 3   Min batch time  6 non-null      int64 
dtypes: int64(1), object(3)
memory usage: 324.0+ bytes


,Product_Id,Flavor,Size,Min batch time
0,OR-600,Orange,600 ml,60
1,LE-600,Lemon lime,600 ml,60
2,CO-600,Cola,600 ml,60
3,DC-600,Diet Cola,600 ml,60
4,RB-600,Root Berry,600 ml,60
5,CO-2L,Cola,2000 ml,98


In [8]:
df_factors.info()
print('number of duplicates :' + str(df_factors.duplicated().sum()))
### cheking coulmns name
##df_factors.columns
df_factors.rename(columns={'Factor': 'Factor_Id'}, inplace=True)
df_factors= df_factors.fillna('fill missing value')
df_factors.isnull().sum()
df_factors.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Factor          12 non-null     int64 
 1   Description     12 non-null     object
 2   Operator Error  12 non-null     object
dtypes: int64(1), object(2)
memory usage: 420.0+ bytes
number of duplicates :0


,Factor_Id,Description,Operator Error
0,1,Emergency stop,No
1,2,Batch change,Yes
2,3,Labeling error,No
3,4,Inventory shortage,No
4,5,Product spill,Yes
5,6,Machine adjustment,Yes
6,7,Machine failure,No
7,8,Batch coding error,Yes
8,9,Conveyor belt jam,No
9,10,Calibration error,Yes


In [9]:
df_downtime.head(2) 
## دا شكل الفايل القديم 

,Batch,1,2,3,4,5,6,7,8,9,10,11,12
0,422111,NaN,60.0,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN
1,422112,NaN,20.0,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN


In [10]:
file_path = 'Manufacturing_Line_Productivity.xlsx'
df_downtime = pd.read_excel(file_path, sheet_name='Line downtime', skiprows=1)
df_melted = df_downtime.melt(
    id_vars=['Batch'], 
    var_name='Downtime factor', 
    value_name='Duration_Minutes'
)
df_downtime_Final = df_melted.dropna(subset=['Duration_Minutes']).copy()
df_downtime_Final = df_downtime_Final.sort_values(by='Batch').reset_index(drop=True)
df_downtime_Final['downtime_id'] = range(1, len(df_downtime_Final) + 1)
df_downtime_Final = df_downtime_Final[['Batch', 'Downtime factor', 'Duration_Minutes', 'downtime_id']]
##df_downtime_Final

In [11]:
df_downtime_Final.info()
print('number of duplicates :' + str(df_downtime_Final.duplicated().sum()))
df_downtime_Final.isnull().sum()
df_downtime_Final.rename(columns={'Batch': 'Batch_Id'}, inplace=True)
df_downtime_Final.rename(columns={'Downtime factor': 'Factor_Id'}, inplace=True)
##df_downtime_Final= df_downtime_Final.fillna('fill missing value')
df_downtime_Final.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Batch             61 non-null     int64  
 1   Downtime factor   61 non-null     object 
 2   Duration_Minutes  61 non-null     float64
 3   downtime_id       61 non-null     int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 2.0+ KB
number of duplicates :0


,Batch_Id,Factor_Id,Duration_Minutes,downtime_id
0,422111,2,60.0,1
1,422111,7,15.0,2
2,422112,2,20.0,3
3,422112,8,20.0,4
4,422113,2,50.0,5


                                       #######Connections########

In [12]:
df_merged= pd.merge(df_productivity, df_products, 
                               on='Product_Id', 
                               how='left')
df_merged.head()

,Date,Product_Id,Batch_Id,Operator_name,Start Time,End Time,Production_duration,shift,Flavor,Size,Min batch time
0,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00,135,Morning,Orange,600 ml,60
1,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00,100,Afternoon,Lemon lime,600 ml,60
2,2024-08-29,LE-600,422113,Mac,15:45:00,17:35:00,110,Afternoon,Lemon lime,600 ml,60
3,2024-08-29,LE-600,422114,Mac,17:35:00,19:15:00,100,Afternoon,Lemon lime,600 ml,60
4,2024-08-29,LE-600,422115,Charlie,19:15:00,20:39:00,84,Afternoon,Lemon lime,600 ml,60


In [13]:
df_downtime_details= pd.merge(df_downtime_Final, df_factors, 
                               on='Factor_Id', 
                               how='left')
df_downtime_details.head()

,Batch_Id,Factor_Id,Duration_Minutes,downtime_id,Description,Operator Error
0,422111,2,60.0,1,Batch change,Yes
1,422111,7,15.0,2,Machine failure,No
2,422112,2,20.0,3,Batch change,Yes
3,422112,8,20.0,4,Batch coding error,Yes
4,422113,2,50.0,5,Batch change,Yes


In [14]:
df_pro_downtime= pd.merge( df_productivity, df_downtime_Final,
                               on='Batch_Id', 
                               how='left')
df_pro_downtime.head()

,Date,Product_Id,Batch_Id,Operator_name,Start Time,End Time,Production_duration,shift,Factor_Id,Duration_Minutes,downtime_id
0,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00,135,Morning,2,60.0,1.0
1,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00,135,Morning,7,15.0,2.0
2,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00,100,Afternoon,2,20.0,3.0
3,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00,100,Afternoon,8,20.0,4.0
4,2024-08-29,LE-600,422113,Mac,15:45:00,17:35:00,110,Afternoon,2,50.0,5.0


In [15]:
complete_table= pd.merge( df_merged,df_downtime_details,
                               on='Batch_Id', 
                               how='left')
complete_table

,Date,Product_Id,Batch_Id,Operator_name,Start Time,End Time,Production_duration,shift,Flavor,Size,Min batch time,Factor_Id,Duration_Minutes,downtime_id,Description,Operator Error
0,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00,135,Morning,Orange,600 ml,60,2,60.0,1.0,Batch change,Yes
1,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00,135,Morning,Orange,600 ml,60,7,15.0,2.0,Machine failure,No
2,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00,100,Afternoon,Lemon lime,600 ml,60,2,20.0,3.0,Batch change,Yes
3,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00,100,Afternoon,Lemon lime,600 ml,60,8,20.0,4.0,Batch coding error,Yes
4,2024-08-29,LE-600,422113,Mac,15:45:00,17:35:00,110,Afternoon,Lemon lime,600 ml,60,2,50.0,5.0,Batch change,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,2024-09-02,CO-2L,422147,Charlie,19:30:00,22:55:00,205,Afternoon,Cola,2000 ml,98,7,30.0,57.0,Machine failure,No
60,2024-09-02,CO-2L,422147,Charlie,19:30:00,22:55:00,205,Afternoon,Cola,2000 ml,98,4,17.0,58.0,Inventory shortage,No
61,2024-09-02,CO-2L,422147,Charlie,19:30:00,22:55:00,205,Afternoon,Cola,2000 ml,98,6,60.0,59.0,Machine adjustment,Yes
62,2024-09-03,CO-2L,422148,Mac,22:55:00,01:05:00,130,Night,Cola,2000 ml,98,4,25.0,60.0,Inventory shortage,No


                                    #####Second version  for documentaion #######

In [16]:
                                     ##Displaying first top 5 row ##
df_productivity.head()
df_products.head()
df_factors.head()
df_downtime_Final.head()

,Batch_Id,Factor_Id,Duration_Minutes,downtime_id
0,422111,2,60.0,1
1,422111,7,15.0,2
2,422112,2,20.0,3
3,422112,8,20.0,4
4,422113,2,50.0,5


In [17]:
##information of tables
df_productivity.info()
df_products.info()
df_factors.info()
df_downtime_Final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 38 non-null     datetime64[ns]
 1   Product_Id           38 non-null     object        
 2   Batch_Id             38 non-null     int64         
 3   Operator_name        38 non-null     object        
 4   Start Time           38 non-null     object        
 5   End Time             38 non-null     object        
 6   Production_duration  38 non-null     int64         
 7   shift                38 non-null     object        
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 2.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Product_Id      6 non-null      object
 1   Flavor          6 non-

In [18]:
df_productivity.isnull().sum()
df_products.isnull().sum()
df_factors.isnull().sum()
df_downtime_Final.isnull().sum()

Batch_Id            0
Factor_Id           0
Duration_Minutes    0
downtime_id         0
dtype: int64

In [19]:
#######fill missing value
# df_productivity= df_productivity.fillna('fill missing value')
# df_factors= df_factors.fillna('fill missing value')
# df_products= df_products.fillna('fill missing value')
# df_factors= df_factors.fillna('fill missing value')
# df_downtime_Final= df_downtime_Final.fillna('fill missing value')

In [20]:
## الوصف هنا ليس له جدوي لان الدتا الي شغالين عليها ليست  numeric 
##df_productivity.describe()
##df_products.describe()
##df_factors.describe()
##df_downtime_Final.describe()

In [21]:
### converting data types 
##df_productivity['Start Time'] = df_productivity['Start Time'].dt.time
##df_productivity['End Time'] = df_productivity['End Time'].dt.time


In [22]:
######formating tables##
#################df_productivity######
##df_productivity['Gross_Minutes'] = (df_productivity['End Time'] - df_productivity['Start Time']).dt.total_seconds() / 60
#################df_products########
##df_products['Size'] = df_products['Size'].replace('2L', '2000 ml')

In [23]:
###Checking Duplication###
print('number of duplicates :' + str(df_productivity.duplicated().sum()))
print('number of duplicates :' + str(df_factors.duplicated().sum()))
print('number of duplicates :' + str(df_downtime_Final.duplicated().sum()))
print('number of duplicates :' + str(df_products.duplicated().sum()))

number of duplicates :0
number of duplicates :0
number of duplicates :0
number of duplicates :0


In [24]:
                               ###Removing Duplication ####
# df_productivity.drop_duplicates(inplace=True)
# df_products.drop_duplicates(inplace=True)
# df_factors.drop_duplicates(inplace=True)
# df_downtime_Final.drop_duplicates(inplace=True)

In [25]:
###future engineering
                            ### df_productivity####
# def get_shift(start_time):
#   hour = start_time.hour
#    if 6 <= hour < 14:
#        return 'Morning'
#    elif 14 <= hour < 22:
#       return 'Evening'
#   else: 
#       return 'Night'
# df_productivity['shift'] = df_productivity['Start Time'].apply(get_shift)
                           ###df_downtime_Final####
# df_downtime_Final['downtime_id'] = range(1, len(df_downtime_Final) + 1)


In [26]:
                         #####renaming columns of tables###########
                                 ##df_productivity##
df_productivity.rename(columns={'Product': 'Product_Id'}, inplace=True)
df_productivity.rename(columns={'Batch': 'Batch_Id'}, inplace=True)
df_productivity.rename(columns={'Operator': 'Operator_name'}, inplace=True)
                                  ##df_products##
df_products.rename(columns={'Product': 'Product_Id'}, inplace=True)
                                  ##df_factors##
df_factors.rename(columns={'Factor': 'Factor_Id'}, inplace=True)
                               ##df_downtime_Final##
df_downtime_Final.rename(columns={'Batch': 'Batch_Id'}, inplace=True)
df_downtime_Final.rename(columns={'Downtime factor': 'Factor_Id'}, inplace=True)

***************************************************analysis****************************************************

In [27]:
complete_table['Batch_Status'] = np.where(
  complete_table['Production_duration'] > complete_table['Min batch time'], 'Delay' , 'On_time'
)
complete_table[['Batch_Id', 'Date', 'Operator_name', 'Start Time', 'End Time', 
    'Min batch time', 'Production_duration', 'Batch_Status', 
    'shift', 'Flavor', 'Size', 'Factor_Id', 'Duration_Minutes', 
    'downtime_id', 'Description', 'Operator Error']]

,Batch_Id,Date,Operator_name,Start Time,End Time,Min batch time,Production_duration,Batch_Status,shift,Flavor,Size,Factor_Id,Duration_Minutes,downtime_id,Description,Operator Error
0,422111,2024-08-29,Mac,11:50:00,14:05:00,60,135,Delay,Morning,Orange,600 ml,2,60.0,1.0,Batch change,Yes
1,422111,2024-08-29,Mac,11:50:00,14:05:00,60,135,Delay,Morning,Orange,600 ml,7,15.0,2.0,Machine failure,No
2,422112,2024-08-29,Mac,14:05:00,15:45:00,60,100,Delay,Afternoon,Lemon lime,600 ml,2,20.0,3.0,Batch change,Yes
3,422112,2024-08-29,Mac,14:05:00,15:45:00,60,100,Delay,Afternoon,Lemon lime,600 ml,8,20.0,4.0,Batch coding error,Yes
4,422113,2024-08-29,Mac,15:45:00,17:35:00,60,110,Delay,Afternoon,Lemon lime,600 ml,2,50.0,5.0,Batch change,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,422147,2024-09-02,Charlie,19:30:00,22:55:00,98,205,Delay,Afternoon,Cola,2000 ml,7,30.0,57.0,Machine failure,No
60,422147,2024-09-02,Charlie,19:30:00,22:55:00,98,205,Delay,Afternoon,Cola,2000 ml,4,17.0,58.0,Inventory shortage,No
61,422147,2024-09-02,Charlie,19:30:00,22:55:00,98,205,Delay,Afternoon,Cola,2000 ml,6,60.0,59.0,Machine adjustment,Yes
62,422148,2024-09-03,Mac,22:55:00,01:05:00,98,130,Delay,Night,Cola,2000 ml,4,25.0,60.0,Inventory shortage,No



                            **************Line Productivity Analysis**************

In [28]:
# 1 Is the number of batches increasing or decreasing over time?
complete_table.groupby('Date')['Batch_Id'].count()

Date
2024-08-29    11
2024-08-30    21
2024-08-31    10
2024-09-02    20
2024-09-03     2
Name: Batch_Id, dtype: int64

In [29]:
# How many batches are completed per shift? (Which shift has the highest production output?)
complete_table.groupby('shift')['Batch_Id'].count()

shift
Afternoon    27
Morning      23
Night        14
Name: Batch_Id, dtype: int64

In [30]:
# # What is the percentage contribution of each product to total production?
product_ratio = complete_table['Product_Id'] \
    .value_counts(normalize=True) \
    .mul(100) \
    .round(2)

print(product_ratio.astype(str) + ' %')

Product_Id
CO-600    39.06 %
RB-600    17.19 %
CO-2L     17.19 %
LE-600    14.06 %
DC-600     9.38 %
OR-600     3.12 %
Name: proportion, dtype: object


In [31]:
# What is the percentage contribution of each product to total production?
batch_mix =complete_table['Product_Id'].value_counts(normalize=True).reset_index()
batch_mix.columns = ['Product_Id', 'Batch_Percentage_%']
batch_mix['Batch_Percentage_%'] = (batch_mix['Batch_Percentage_%'] * 100).round(2)

time_mix =complete_table.groupby('Product_Id')['Duration_Minutes'].sum().reset_index()
total_time = time_mix['Duration_Minutes'].sum()
time_mix['Time_Percentage_%'] = ((time_mix['Duration_Minutes'] / total_time) * 100).round(2)

result = pd.merge(batch_mix, time_mix[['Product_Id', 'Time_Percentage_%']], on='Product_Id')

print(result)


  Product_Id  Batch_Percentage_%  Time_Percentage_%
0     CO-600               39.06              35.59
1     RB-600               17.19              18.59
2      CO-2L               17.19              19.96
3     LE-600               14.06              12.18
4     DC-600                9.38               8.29
5     OR-600                3.12               5.40


In [32]:
# What is the difference between total actual production time and ideal production time?
total_actual = complete_table['Duration_Minutes'].sum()
total_ideal = complete_table['Min batch time'].sum()

time_loss = total_actual - total_ideal
print(f" total_actual: {total_actual}")
print(f" total_ideal: {total_ideal}")
print(f" time_loss: {time_loss}")

 total_actual: 1388.0
 total_ideal: 4258
 time_loss: -2870.0


In [33]:
# How many hours are lost due to inefficiencies?
utilization = total_actual / (24 * 60 * complete_table['Date'].nunique())
utilization = utilization * 100
print(f" {utilization:.2f}%")

 19.28%


In [34]:
#Which batch took the longest time (Outlier), and why? Which was the fastest batch completed? 

longest_batch = complete_table.sort_values(by='Duration_Minutes', ascending=False).iloc[0]
shortest_batch = complete_table.sort_values(by='Duration_Minutes', ascending=True).iloc[0]


print("(Longest Batch Outlier):")
print(f"(Batch_Id): {longest_batch['Batch_Id']}")
print(f"(Duration_Minutes): {longest_batch['Duration_Minutes']} ")
print(f" (Description): {longest_batch['Description']}")

print("\n" + "="*40 + "\n") 

print("(Shortest Batch):")
print(f" (Batch_Id): {shortest_batch['Batch_Id']}")
print(f" (Duration_Minutes): {shortest_batch['Duration_Minutes']} ")
print(f" (Description): {shortest_batch['Description']}")

(Longest Batch Outlier):
(Batch_Id): 422111
(Duration_Minutes): 60.0 
 (Description): Batch change


(Shortest Batch):
 (Batch_Id): 422117
 (Duration_Minutes): 5.0 
 (Description): Machine adjustment


In [60]:
#How many times was the product changed per day?
complete_table['Product_Change'] = complete_table['Product_Id'].ne(
    complete_table['Product_Id'].shift()
)

changeovers = complete_table.groupby('Date')['Product_Change'].sum() 
print(changeovers)

Date
2024-08-29    2
2024-08-30    1
2024-08-31    1
2024-09-02    2
2024-09-03    0
Name: Product_Change, dtype: int64


In [63]:
##What is the time gap between the End Time of one batch and the Start Time of the next batch?
##(This reveals possible idle or non-productive time between operations.) 

complete_table = complete_table.sort_values(by=['Date', 'Start Time'])
start_dt = pd.to_datetime(complete_table['Date'].astype(str) + ' ' + complete_table['Start Time'].astype(str))
end_dt = pd.to_datetime(complete_table['Date'].astype(str) + ' ' + complete_table['End Time'].astype(str))
complete_table['Idle_Time'] = (start_dt - end_dt.shift()).dt.total_seconds() / 60

complete_table['Idle_Time'] = complete_table['Idle_Time'].fillna(0)

print("Idle Time Gap Analysis Fixed:")
display(complete_table[['Date', 'Batch_Id', 'Start Time', 'End Time', 'Idle_Time']].head(10))

Idle Time Gap Analysis Fixed:


,Date,Batch_Id,Start Time,End Time,Idle_Time
0,2024-08-29,422111,11:50:00,14:05:00,0.0
1,2024-08-29,422111,11:50:00,14:05:00,-135.0
2,2024-08-29,422112,14:05:00,15:45:00,0.0
3,2024-08-29,422112,14:05:00,15:45:00,-100.0
4,2024-08-29,422113,15:45:00,17:35:00,0.0
5,2024-08-29,422114,17:35:00,19:15:00,0.0
6,2024-08-29,422114,17:35:00,19:15:00,-100.0
7,2024-08-29,422115,19:15:00,20:39:00,0.0
8,2024-08-29,422116,20:39:00,21:39:00,0.0
9,2024-08-29,422117,21:39:00,22:54:00,0.0


In [64]:
#What is the total number of batches (Total Batches) for each product? 

total_batches_per_product = complete_table.groupby('Product_Id')['Batch_Id'].nunique().sort_values(ascending=False).reset_index()

total_batches_per_product.columns = ['Product_Id', 'Total_Batches']

print("Total number of batches per product:")
display(total_batches_per_product)

Total number of batches per product:


,Product_Id,Total_Batches
0,CO-600,15
1,RB-600,7
2,LE-600,6
3,CO-2L,5
4,DC-600,4
5,OR-600,1


In [65]:
##What is the average actual production duration for each product? 
avg_actual_duration = complete_table.groupby('Product_Id')['Production_duration'].mean().sort_values(ascending=False).reset_index()

avg_actual_duration.columns = ['Product_Id', 'Average_Actual_Duration']

print("Average actual production time per product (in minutes):")
display(avg_actual_duration.round(2))

Average actual production time per product (in minutes):


,Product_Id,Average_Actual_Duration
0,CO-2L,161.73
1,OR-600,135.00
2,RB-600,101.73
3,CO-600,99.72
4,DC-600,95.00
5,LE-600,89.33


In [66]:
##How many times was the status marked as ON TIME versus DELAY? (Percentage ratio)  
status_counts = complete_table.groupby('Batch_Status')['Batch_Id'].nunique()
total_batches = complete_table['Batch_Id'].nunique()
status_ratio = (status_counts / total_batches) * 100

status_analysis = pd.DataFrame({
    'Count': status_counts,
    'Percentage (%)': status_ratio.round(2)
})

print(f"Total number of batches: {total_batches}")
display(status_analysis)

Total number of batches: 38


,Count,Percentage (%)
Batch_Status,,
Delay,35,92.11
On_time,3,7.89


In [67]:
##Which day recorded the highest production volume? 
daily_volume = complete_table.groupby('Date')['Batch_Id'].nunique().sort_values(ascending=False).reset_index()
daily_volume.columns = ['Date', 'Batch_Count']
peak_day = daily_volume.iloc[0]

print(f"The day with the highest production volume is: {peak_day['Date'].date()} with a total of {peak_day['Batch_Count']} batches.")
print("Production volume details for all days:")
display(daily_volume)

The day with the highest production volume is: 2024-08-30 with a total of 12 batches.
Production volume details for all days:


,Date,Batch_Count
0,2024-08-30,12
1,2024-09-02,11
2,2024-08-29,7
3,2024-08-31,7
4,2024-09-03,1


In [68]:
##What is the difference between Actual Production Duration and Ideal Time (Min_Batch_Time) for each batch? 
complete_table['Time_Deviation'] = complete_table['Production_duration'] - complete_table['Min batch time']
print("Difference between actual and ideal time for each batch (in minutes):")
display(complete_table[['Batch_Id', 'Product_Id', 'Min batch time', 'Production_duration', 'Time_Deviation']].head(10))
avg_delay = complete_table['Time_Deviation'].mean()
print(f"\nAverage delay per batch in line: {avg_delay: .2f} minutes")

Difference between actual and ideal time for each batch (in minutes):


,Batch_Id,Product_Id,Min batch time,Production_duration,Time_Deviation
0,422111,OR-600,60,135,75
1,422111,OR-600,60,135,75
2,422112,LE-600,60,100,40
3,422112,LE-600,60,100,40
4,422113,LE-600,60,110,50
5,422114,LE-600,60,100,40
6,422114,LE-600,60,100,40
7,422115,LE-600,60,84,24
8,422116,LE-600,60,60,0
9,422117,LE-600,60,75,15



Average delay per batch in line:  43.39 minutes


In [69]:
##Which shift (Morning, Afternoon, Night) has the highest delay rate versus ideal time? 
if 'Time_Deviation' not in complete_table.columns:
    complete_table['Time_Deviation'] = complete_table['Production_duration'] - complete_table['Min batch time']
shift_delay_analysis = complete_table.groupby('shift')['Time_Deviation'].mean().sort_values(ascending=False).reset_index()
shift_status_counts = complete_table.groupby(['shift', 'Batch_Status'])['Batch_Id'].nunique().unstack(fill_value=0)
shift_status_counts['Total_Batches'] = shift_status_counts.sum(axis=1)
shift_status_counts['Delay_Rate_%'] = (shift_status_counts['Delay'] / shift_status_counts['Total_Batches']) * 100
final_shift_analysis = pd.merge(shift_delay_analysis, shift_status_counts[['Delay_Rate_%']].reset_index(), on='shift')
final_shift_analysis.columns = ['Shift', 'Average_Delay_Minutes', 'Delay_Percentage_%']

print("Analysis of lateness rates by shift:")
display(final_shift_analysis.round(2))

Analysis of lateness rates by shift:


,Shift,Average_Delay_Minutes,Delay_Percentage_%
0,Night,44.64,100.00
1,Afternoon,43.48,87.50
2,Morning,42.52,93.33


In [78]:
##	Do large-size products (2L) experience higher delays than small-size bottles (600ml)? 

size_delay_avg = complete_table.groupby('Size')['Time_Deviation'].mean().reset_index()


size_status = complete_table.groupby(['Size', 'Batch_Status'])['Batch_Id'].nunique().unstack(fill_value=0)
size_status['Total'] = size_status.sum(axis=1)
size_status['Delay_Rate_%'] = (size_status['Delay'] / size_status['Total']) * 100


comparison = pd.merge(size_delay_avg, size_status[['Delay_Rate_%']].reset_index(), on='Size')
comparison.columns = ['Size', 'Avg_Delay_Minutes', 'Delay_Percentage_%']

print("Comparison of delay between large (2L) and small (600ml) sizes:")
display(comparison.round(2))

Comparison of delay between large (2L) and small (600ml) sizes:


,Size,Avg_Delay_Minutes,Delay_Percentage_%
0,2000 ml,63.73,100.00
1,600 ml,39.17,90.91


In [81]:
##Do operators tend to have more delays toward the end of the shift compared to the beginning? 

unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()

unique_batches = unique_batches.sort_values(by=['Date', 'Operator_name', 'Start Time'])
unique_batches['Batch_Sequence'] = unique_batches.groupby(['Date', 'Operator_name']).cumcount() + 1

sequence_impact = unique_batches.groupby('Batch_Sequence')['Time_Deviation'].mean().reset_index()
sequence_impact.columns = ['Batch order per day', 'Average delay (minutes)']

print("The impact of advancing working hours on operator efficiency:")
display(sequence_impact.round(2))

The impact of advancing working hours on operator efficiency:


,Batch order per day,Average delay (minutes)
0,1,33.64
1,2,32.10
2,3,41.30
3,4,44.83
4,5,15.00


In [82]:
##(Min_Batch_Time/Actual Duration)×100(Min\_Batch\_Time / Actual\ Duration) \times 100(Min_Batch_Time/Actual Duration)×100 
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()

total_min_time = unique_batches['Min batch time'].sum()
total_actual_duration = unique_batches['Production_duration'].sum()
overall_efficiency = (total_min_time / total_actual_duration) * 100

print(f"Overall_efficiency of the line: {overall_efficiency:.2f}%")

operator_eff = unique_batches.groupby('Operator_name').agg({
    'Min batch time': 'sum',
    'Production_duration': 'sum'
}).reset_index()

operator_eff['Efficiency_%'] = (operator_eff['Min batch time'] / operator_eff['Production_duration']) * 100
operator_eff = operator_eff.sort_values(by='Efficiency_%', ascending=False)

print("\nProduction efficiency by operator (from highest to lowest):")
display(operator_eff.round(2))

Overall_efficiency of the line: 64.02%

Production efficiency by operator (from highest to lowest):


,Operator_name,Min batch time,Production_duration,Efficiency_%
0,Charlie,774,1158,66.84
1,Dee,660,1030,64.08
2,Dennis,518,820,63.17
3,Mac,518,850,60.94


In [83]:
##•	Average delay minutes per batch. 
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()

unique_batches['Delay_Minutes'] = unique_batches['Production_duration'] - unique_batches['Min batch time']

overall_avg_delay = unique_batches['Delay_Minutes'].mean()

print(f"Average minutes delay per batch (general): {overall_avg_delay:.2f} minutes")

avg_delay_by_product = unique_batches.groupby('Product_Id')['Delay_Minutes'].mean().sort_values(ascending=False)
avg_delay_by_operator = unique_batches.groupby('Operator_name')['Delay_Minutes'].mean().sort_values(ascending=False)

print("\nAverage delay by product type (in minutes):")
print(avg_delay_by_product.round(2))

print("\nAverage delay by operator (in minutes):")
print(avg_delay_by_operator.round(2))

Average minutes delay per batch (general): 36.53 minutes

Average delay by product type (in minutes):
Product_Id
OR-600    75.00
CO-2L     55.40
RB-600    36.86
CO-600    32.93
DC-600    28.75
LE-600    28.17
Name: Delay_Minutes, dtype: float64

Average delay by operator (in minutes):
Operator_name
Mac        41.50
Dennis     37.75
Charlie    34.91
Dee        33.64
Name: Delay_Minutes, dtype: float64


In [84]:
##Percentage of batches completed on time or earlier. 

unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()

total_batches = unique_batches['Batch_Id'].nunique()
on_time_count = unique_batches[unique_batches['Batch_Status'] == 'On_time']['Batch_Id'].nunique()

on_time_percentage = (on_time_count / total_batches) * 100

print(f"Total number of batches: {total_batches}")
print(f"Number of batches completed on time: {on_time_count}")
print(f"Percentage of batches completed on time or less: {on_time_percentage: .2f}%")

Total number of batches: 38
Number of batches completed on time: 3
Percentage of batches completed on time or less:  7.89%


In [85]:
##Number of liters or units produced per hour based on size and production duration. 
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()

unique_batches['Size_Numeric'] = unique_batches['Size'].str.extract('(\d+)').astype(float)

unique_batches['Units_Per_Hour'] = 60 / unique_batches['Production_duration']
unique_batches['Liters_Per_Hour'] = (unique_batches['Size_Numeric'] / 1000) * unique_batches['Units_Per_Hour']

production_rates = unique_batches.groupby('Product_Id').agg({
    'Units_Per_Hour': 'mean',
    'Liters_Per_Hour': 'mean'
}).reset_index()

print("Production rate (units and liters) per hour per product:")
display(production_rates.round(2).sort_values(by='Liters_Per_Hour', ascending=False))

Production rate (units and liters) per hour per product:


,Product_Id,Units_Per_Hour,Liters_Per_Hour
0,CO-2L,0.40,0.81
2,DC-600,0.72,0.43
3,LE-600,0.71,0.43
1,CO-600,0.67,0.40
5,RB-600,0.64,0.39
4,OR-600,0.44,0.27


In [86]:
##Each product’s share of total production. 
batch_share = complete_table['Product_Id'].value_counts(normalize=True).mul(100).round(2).reset_index()
batch_share.columns = ['Product_Id', 'Batch_Percentage_%']
total_production_time = complete_table['Duration_Minutes'].sum()
time_share = complete_table.groupby('Product_Id')['Duration_Minutes'].sum().reset_index()
time_share['Time_Percentage_%'] = (time_share['Duration_Minutes'] / total_production_time).mul(100).round(2)

production_share = pd.merge(batch_share, time_share[['Product_Id', 'Time_Percentage_%']], on='Product_Id')

print("Each product's share of total production (number of batches vs. time):")
display(production_share)

Each product's share of total production (number of batches vs. time):


,Product_Id,Batch_Percentage_%,Time_Percentage_%
0,CO-600,39.06,35.59
1,RB-600,17.19,18.59
2,CO-2L,17.19,19.96
3,LE-600,14.06,12.18
4,DC-600,9.38,8.29
5,OR-600,3.12,5.40


In [87]:
##What is the average delay time for batches marked DELAY only? 
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()

unique_batches['Delay_Minutes'] = unique_batches['Production_duration'] - unique_batches['Min batch time']
avg_delay_only = unique_batches[unique_batches['Batch_Status'] == 'Delay']['Delay_Minutes'].mean()

print(f"Average delay time for late patches only: {avg_delay_only:.2f} minutes")

Average delay time for late patches only: 39.66 minutes


                                                                   Downtime Analysis

In [77]:
##What is the most frequent downtime factor? (By occurrence count) 
frequent_factors = complete_table['Description'].value_counts().reset_index()
frequent_factors.columns = ['Downtime Factor', 'Occurrence Count']

total_incidents = frequent_factors['Occurrence Count'].sum()
frequent_factors['Frequency_%'] = (frequent_factors['Occurrence Count'] / total_incidents) * 100

print("Order causes of failures by number of times they occur:")
print(frequent_factors.round(2))

top_factor = frequent_factors.iloc[0]['Downtime Factor']
top_count = frequent_factors.iloc[0]['Occurrence Count']
print(f"\n The most frequent cause is: '{top_factor}' where {top_count} happened once.")

Order causes of failures by number of times they occur:
       Downtime Factor  Occurrence Count  Frequency_%
0   Machine adjustment                12        19.67
1      Machine failure                11        18.03
2   Inventory shortage                 9        14.75
3   Batch coding error                 6         9.84
4                Other                 6         9.84
5         Batch change                 5         8.20
6    Calibration error                 3         4.92
7         Label switch                 3         4.92
8        Product spill                 3         4.92
9       Labeling error                 2         3.28
10   Conveyor belt jam                 1         1.64

=> The most frequent cause is: 'Machine adjustment' where 12 happened once.


In [78]:
##•	Which factor consumes the highest downtime duration? (Total minutes) 

total_duration_by_factor = complete_table.groupby('Description')['Duration_Minutes'].sum().reset_index()

total_duration_by_factor = total_duration_by_factor.sort_values(by='Duration_Minutes', ascending=False).reset_index(drop=True)

print("Ranking causes of failures by total minutes wasted (Total Duration):")
print(total_duration_by_factor)

top_duration_factor = total_duration_by_factor.iloc[0]['Description']
top_minutes = total_duration_by_factor.iloc[0]['Duration_Minutes']
print(f"\n The 'most time consuming' factor is: {top_duration_factor} with a total of {top_minutes} minutes.")

Ranking causes of failures by total minutes wasted (Total Duration):
           Description  Duration_Minutes
0   Machine adjustment             332.0
1      Machine failure             254.0
2   Inventory shortage             225.0
3         Batch change             160.0
4   Batch coding error             145.0
5                Other              74.0
6        Product spill              57.0
7    Calibration error              49.0
8       Labeling error              42.0
9         Label switch              33.0
10   Conveyor belt jam              17.0

 The 'most time consuming' factor is: Machine adjustment with a total of 332.0 minutes.


In [79]:
##•	What is the total downtime for each shift? 
shift_downtime_total = complete_table.groupby('shift')['Duration_Minutes'].sum().reset_index()
shift_downtime_total = shift_downtime_total.sort_values(by='Duration_Minutes', ascending=False).reset_index(drop=True)
print("Total crash time (in minutes) per shift:")
print(shift_downtime_total)
total_factory_downtime = shift_downtime_total['Duration_Minutes'].sum()
shift_downtime_total['Share_%'] = (shift_downtime_total['Duration_Minutes'] / total_factory_downtime) * 100

print("\nDistribution of failure rates across shifts:")
print(shift_downtime_total[['shift', 'Share_%']].round(2))

Total crash time (in minutes) per shift:
       shift  Duration_Minutes
0  Afternoon             584.0
1    Morning             534.0
2      Night             270.0

Distribution of failure rates across shifts:
       shift  Share_%
0  Afternoon    42.07
1    Morning    38.47
2      Night    19.45


In [80]:
##•	What is the total downtime for each day? 
daily_downtime = complete_table.groupby('Date')['Duration_Minutes'].sum().reset_index()
daily_downtime['Date'] = pd.to_datetime(daily_downtime['Date'])
daily_downtime = daily_downtime.sort_values(by='Date')
print("Total crash time (in minutes) per day:")
print(daily_downtime)

avg_daily = daily_downtime['Duration_Minutes'].mean()
print(f"\n Average daily crash time: {avg_daily: .2f} minutes.")

Total crash time (in minutes) per day:
        Date  Duration_Minutes
0 2024-08-29             244.0
1 2024-08-30             444.0
2 2024-08-31             165.0
3 2024-09-02             503.0
4 2024-09-03              32.0

 Average daily crash time:  277.60 minutes.


In [81]:
##What percentage of failures falls under Operator Error versus machine failures? 
total_incidents = len(complete_table)
operator_error_count = len(complete_table[complete_table['Operator Error'] == 'Yes'])
machine_failure_count = len(complete_table[complete_table['Description'] == 'Machine failure'])
operator_error_pct = (operator_error_count / total_incidents) * 100
machine_failure_pct = (machine_failure_count / total_incidents) * 100
other_errors_pct = 100 - (operator_error_pct + machine_failure_pct)
failure_dist = pd.DataFrame({
    'Failure Category': ['Operator Error', 'Machine Failure', 'Other Factors'],
    'Count': [operator_error_count, machine_failure_count, total_incidents - (operator_error_count + machine_failure_count)],
    'Percentage_%': [operator_error_pct, machine_failure_pct, other_errors_pct]
})

print("Distribution of faults: worker faults vs. machine faults:")
print(failure_dist.round(2))

Distribution of faults: worker faults vs. machine faults:
  Failure Category  Count  Percentage_%
0   Operator Error     32         50.00
1  Machine Failure     11         17.19
2    Other Factors     21         32.81


In [82]:
###Which 20% of causes are responsible for 80% of lost time? 
pareto_data = complete_table.groupby('Description')['Duration_Minutes'].sum().sort_values(ascending=False).reset_index()

# 2. حساب النسبة المئوية والنسبة التراكمية (Cumulative Percentage)
total_lost_time = pareto_data['Duration_Minutes'].sum()
pareto_data['Percentage'] = (pareto_data['Description'].apply(lambda x: (pareto_data[pareto_data['Description'] == x]['Duration_Minutes'].sum() / total_lost_time) * 100)) # Calculate individual %
pareto_data['Cum_Percentage'] = (pareto_data['Duration_Minutes'].cumsum() / total_lost_time) * 100
vital_few = pareto_data[pareto_data['Cum_Percentage'] <= 85] # استخدمنا 85 لتغطية النطاق القريب من الـ 80%

print("Main causes responsible for the majority of wasted time (Vital Few):")
print(vital_few[['Description', 'Duration_Minutes', 'Cum_Percentage']].round(2))

Main causes responsible for the majority of wasted time (Vital Few):
          Description  Duration_Minutes  Cum_Percentage
0  Machine adjustment             332.0           23.92
1     Machine failure             254.0           42.22
2  Inventory shortage             225.0           58.43
3        Batch change             160.0           69.96
4  Batch coding error             145.0           80.40


In [83]:
##Are certain products (such as CO-2L) associated with more Batch Change or Label Switch failures? 
target_factors = [2, 3]
change_label_issues = complete_table[complete_table['Factor_Id'].isin(target_factors)]
product_issue_matrix = change_label_issues.groupby(['Product_Id', 'Description']).size().unstack(fill_value=0)
product_issue_matrix['Total_Issues'] = product_issue_matrix.sum(axis=1)
print("Analysis of product association with patch change malfunctions and label errors:")
print(product_issue_matrix.sort_values(by='Total_Issues', ascending=False))
if 'CO-2L' in product_issue_matrix.index:
    co2l_data = product_issue_matrix.loc['CO-2L']
    print(f"\n CO-2L product-specific analysis:")
    print(f"  Number of patch change failures: {co2l_data.get('Batch change', 0)}")
    print(f"  Number of label errors: {co2l_data.get('Labeling error', 0)}")

Analysis of product association with patch change malfunctions and label errors:
Description  Batch change  Labeling error  Total_Issues
Product_Id                                             
LE-600                  3               0             3
CO-2L                   0               1             1
CO-600                  1               0             1
OR-600                  1               0             1
RB-600                  0               1             1

 CO-2L product-specific analysis:
  Number of patch change failures: 0
  Number of label errors: 1


In [84]:
##Are certain operators associated with repeated Operator Errors (e.g. Product Spill or Calibration Error)? 
operator_faults = complete_table[complete_table['Operator Error'] == 'Yes'].copy()
operator_error_matrix = operator_faults.groupby(['Operator_name', 'Description']).size().unstack(fill_value=0)
operator_error_matrix['Total_Errors'] = operator_error_matrix.sum(axis=1)
operator_error_matrix = operator_error_matrix.sort_values(by='Total_Errors', ascending=False)
print("Analysis of recurring human errors by operator and fault type:")
print(operator_error_matrix)
if not operator_error_matrix.empty:
    top_operator = operator_error_matrix.index[0]
    top_error_count = operator_error_matrix.iloc[0]['Total_Errors']
    print(f"\nThe operator most associated with human errors is: {top_operator} with total {top_error_count} errors.")

Analysis of recurring human errors by operator and fault type:
Description    Batch change  Batch coding error  Calibration error  \
Operator_name                                                        
Dee                       1                   1                  2   
Charlie                   1                   1                  1   
Mac                       3                   3                  0   
Dennis                    0                   1                  0   

Description    Label switch  Machine adjustment  Product spill  Total_Errors  
Operator_name                                                                 
Dee                       2                   3              1            10  
Charlie                   1                   4              1             9  
Mac                       0                   1              0             7  
Dennis                    0                   4              1             6  

The operator most associated with human e

In [85]:
##Is downtime duration higher in the Night Shift compared to Morning Shift?
##(This may indicate lack of technical support or maintenance at night.) 
shift_downtime_analysis = complete_table.groupby('shift')['Duration_Minutes'].agg(['sum', 'mean', 'count']).reset_index()
shift_downtime_analysis.columns = ['Shift', 'Total_Downtime_Mins', 'Avg_Downtime_Per_Incident', 'Incident_Count']
morning_vs_night = shift_downtime_analysis[shift_downtime_analysis['Shift'].isin(['Morning', 'Night'])]

print("Comparing morning shift versus night shift malfunctions:")
print(morning_vs_night.round(2))
night_avg = morning_vs_night[morning_vs_night['Shift'] == 'Night']['Avg_Downtime_Per_Incident'].values[0]
morning_avg = morning_vs_night[morning_vs_night['Shift'] == 'Morning']['Avg_Downtime_Per_Incident'].values[0]

diff = night_avg - morning_avg
print(f"\n The average duration of one holiday per night shift is {diff:.2f} minutes greater than the morning shift.")


Comparing morning shift versus night shift malfunctions:
     Shift  Total_Downtime_Mins  Avg_Downtime_Per_Incident  Incident_Count
1  Morning                534.0                      24.27              22
2    Night                270.0                      19.29              14

 The average duration of one holiday per night shift is -4.99 minutes greater than the morning shift.


In [86]:
##•	How many minutes are lost due to Batch Change setup time? 
total_batch_change_minutes = complete_table[complete_table['Factor_Id'] == 2]['Duration_Minutes'].sum()

print(f"Total minutes lost due to patch change processing time: {total_batch_change_minutes} minutes.")

Total minutes lost due to patch change processing time: 160.0 minutes.


In [87]:
##Is this time consistent, or does it increase in certain shifts? 
setup_data = complete_table[complete_table['Factor_Id'] == 2]
setup_consistency = setup_data.groupby('shift')['Duration_Minutes'].agg(['count', 'mean', 'std']).reset_index()
setup_consistency.columns = ['Shift', 'Changeover_Count', 'Average_Minutes', 'Standard_Deviation']
print("Pitch Change Preparation Time Consistency Analysis by Shift:")
print(setup_consistency.round(2))
if not setup_consistency.empty:
    max_shift = setup_consistency.loc[setup_consistency['Average_Minutes'].idxmax()]
    min_shift = setup_consistency.loc[setup_consistency['Average_Minutes'].idxmin()]
    diff = max_shift['Average_Minutes'] - min_shift['Average_Minutes']
    
    print(f"\n The difference between the fastest and slowest shift in processing is {diff:.2f} minutes.")
    print(f" The slowest processing shift is: {max_shift['Shift']}.")

Pitch Change Preparation Time Consistency Analysis by Shift:
       Shift  Changeover_Count  Average_Minutes  Standard_Deviation
0  Afternoon                 3            26.67               20.82
1    Morning                 2            40.00               28.28

 The difference between the fastest and slowest shift in processing is 13.33 minutes.
 The slowest processing shift is: Morning.


In [88]:
##•	Is there a relationship between certain downtime factors and DELAY status? 
factor_delay_relation = complete_table.groupby(['Description', 'Batch_Status']).size().unstack(fill_value=0)
factor_delay_relation['Total_Occurrences'] = factor_delay_relation.sum(axis=1)
factor_delay_relation['Delay_Probability_%'] = (factor_delay_relation['Delay'] / factor_delay_relation['Total_Occurrences']) * 100
print("Relationship between fault type and delay state (Delay):")
print(factor_delay_relation.sort_values(by='Delay_Probability_%', ascending=False).round(2))

Relationship between fault type and delay state (Delay):
Batch_Status        Delay  Total_Occurrences  Delay_Probability_%
Description                                                      
Batch change            5                  5                100.0
Batch coding error      6                  6                100.0
Calibration error       3                  3                100.0
Conveyor belt jam       1                  1                100.0
Inventory shortage      9                  9                100.0
Label switch            3                  3                100.0
Labeling error          2                  2                100.0
Machine adjustment     12                 12                100.0
Machine failure        11                 11                100.0
Other                   6                  6                100.0
Product spill           3                  3                100.0


In [89]:
##•	Are specific products (e.g. RB-600) repeatedly associated with Labeling Errors or Calibration Errors? 
target_descriptions = ['Labeling error', 'Calibration Error', 'Machine adjustment']

specific_faults = complete_table[complete_table['Description'].isin(target_descriptions)]
product_fault_matrix = specific_faults.groupby(['Product_Id', 'Description']).size().unstack(fill_value=0)
product_fault_matrix['Total_Specific_Errors'] = product_fault_matrix.sum(axis=1)
print("Analysis of product correlation with labeling and calibration errors:")
print(product_fault_matrix.sort_values(by='Total_Specific_Errors', ascending=False))
if 'RB-600' in product_fault_matrix.index:
    rb_data = product_fault_matrix.loc['RB-600']
    print(f"\n RB-600 Product Analysis:")
    print(f" Label errors: {rb_data.get('Labeling error', 0)}")
    print(f" Calibration/tuning errors: {rb_data.get('Calibration Error', 0) + rb_data.get('Machine adjustment', 0)}")

Analysis of product correlation with labeling and calibration errors:
Description  Labeling error  Machine adjustment  Total_Specific_Errors
Product_Id                                                            
RB-600                    1                   4                      5
CO-2L                     1                   3                      4
CO-600                    0                   3                      3
LE-600                    0                   2                      2

 RB-600 Product Analysis:
  Label errors: 1
 Calibration/tuning errors: 4


In [90]:
##•	Is Night Shift associated with higher Machine Failure incidents? 
machine_failures_only = complete_table[complete_table['Description'] == 'Machine failure']
shift_failure_counts = machine_failures_only.groupby('shift').size().reset_index(name='Incident_Count')
total_batches = df_productivity['shift'].value_counts().reset_index()
total_batches.columns = ['shift', 'Total_Batches']

failure_rate_analysis = pd.merge(shift_failure_counts, total_batches, on='shift')
failure_rate_analysis['Failure_Rate_%'] = (failure_rate_analysis['Incident_Count'] / failure_rate_analysis['Total_Batches']) * 100
print("Analysis of machine failure rate by shift:")
print(failure_rate_analysis.sort_values(by='Failure_Rate_%', ascending=False).round(2))

Analysis of machine failure rate by shift:
       shift  Incident_Count  Total_Batches  Failure_Rate_%
1    Morning               7             15           46.67
0  Afternoon               3             16           18.75
2      Night               1              7           14.29


In [91]:
##Average time required to resolve a downtime event.
avg_resolution_time = complete_table.groupby('Description')['Duration_Minutes'].mean().reset_index()
avg_resolution_time.columns = ['Downtime Factor', 'Avg Resolution Time (Mins)']
avg_resolution_time = avg_resolution_time.sort_values(by='Avg Resolution Time (Mins)', ascending=False)
print("Average time taken to resolve each fault type (in minutes):")
print(avg_resolution_time.round(2).reset_index(drop=True))
overall_avg = complete_table['Duration_Minutes'].mean()
print(f"\n The overall average resolution time for any line fault is: {overall_avg: .2f} minutes.")

Average time taken to resolve each fault type (in minutes):
       Downtime Factor  Avg Resolution Time (Mins)
0         Batch change                       32.00
1   Machine adjustment                       27.67
2   Inventory shortage                       25.00
3   Batch coding error                       24.17
4      Machine failure                       23.09
5       Labeling error                       21.00
6        Product spill                       19.00
7    Conveyor belt jam                       17.00
8    Calibration error                       16.33
9                Other                       12.33
10        Label switch                       11.00

 The overall average resolution time for any line fault is:  22.75 minutes.


In [92]:
##Total Downtime÷Total Production Time×100Total\ Downtime \div Total\ Production\ Time \times 100Total Downtime÷Total Production Time×100 
total_downtime_minutes = complete_table['Duration_Minutes'].sum()
total_production_time = df_productivity['Production_duration'].sum()
downtime_ratio = (total_downtime_minutes / total_production_time) * 100
print(f"Total crash minutes: {total_downtime_minutes:,.2f} minutes")
print(f"Total production time: {total_production_time:,.2f} minutes")
print(f"Downtime Ratio (Downtime Ratio): {downtime_ratio: .2f}%")

Total crash minutes: 1,388.00 minutes
Total production time: 3,858.00 minutes
Downtime Ratio (Downtime Ratio):  35.98%


In [93]:
##100%−Downtime Ratio100\% - Downtime\ Ratio100%−Downtime Ratio 
total_downtime_minutes = complete_table['Duration_Minutes'].sum()
total_production_time = df_productivity['Production_duration'].sum()

downtime_ratio = (total_downtime_minutes / total_production_time) * 100

uptime_ratio = 100 - downtime_ratio

print(f"Downtime Ratio (Downtime Ratio): {downtime_ratio: .2f}%")
print(f"Actual run ratio (Uptime Ratio): {uptime_ratio:.2f}%")


Downtime Ratio (Downtime Ratio):  35.98%
Actual run ratio (Uptime Ratio): 64.02%


In [88]:
##Based on Min_Batch_Time, how many batches were lost due to downtime? 
total_downtime_minutes = complete_table['Duration_Minutes'].sum()
avg_min_batch_time = complete_table['Min batch time'].mean()
lost_batches_count = total_downtime_minutes / avg_min_batch_time

print(f"Total downtime minutes: {total_downtime_minutes:,.2f} minutes")
print(f"Average optimal time to produce one batch: {avg_min_batch_time:.2f} minutes")
print(f"Number of batches lost due to these errors: {lost_batches_count:.2f} batches")

Total downtime minutes: 1,388.00 minutes
Average optimal time to produce one batch: 66.53 minutes
Number of batches lost due to these errors: 20.86 batches


In [94]:
##Average operating time between failures. 
total_production_duration = df_productivity['Production_duration'].sum()
total_downtime_duration = complete_table['Duration_Minutes'].sum()
total_operating_time = total_production_duration - total_downtime_duration
actual_failures = complete_table[complete_table['Factor_Id'] != 2]
failure_count = len(actual_failures)
if failure_count > 0:
    mtbf = total_operating_time / failure_count
else:
    mtbf = total_operating_time
print(f"Total actual runtime: {total_operating_time:,.2f} minutes")
print(f"Number of sudden failures: {failure_count}")
print(f"Mean run time between failures (MTBF): {mtbf: .2f} minutes")

Total actual runtime: 2,470.00 minutes
Number of sudden failures: 59
Mean run time between failures (MTBF):  41.86 minutes


In [95]:
##Percentage of downtime minutes caused by human errors. 
total_downtime_minutes = complete_table['Duration_Minutes'].sum()
human_error_minutes = complete_table[complete_table['Operator Error'] == 'Yes']['Duration_Minutes'].sum()
human_error_percentage = (human_error_minutes / total_downtime_minutes) * 100
print(f"Total Stop Minutes: {total_downtime_minutes:,.2f} minutes")
print(f"minutes stopped due to human errors: {human_error_minutes:,.2f} minutes")
print(f"Percentage of human errors from downtime: {human_error_percentage: .2f}%")

Total Stop Minutes: 1,388.00 minutes
minutes stopped due to human errors: 776.00 minutes
Percentage of human errors from downtime:  55.91%


In [92]:
##Do all DELAY batches have recorded downtime causes?
##If not, there may be hidden lost time. 
delayed_batches = complete_table[complete_table['Batch_Status'] == 'Delay'].copy()

delayed_batches['Has_Downtime_Reason'] = delayed_batches['Description'].notnull() & \
                                          (delayed_batches['Description'] != 'fill missing value')

missing_reasons = delayed_batches[delayed_batches['Has_Downtime_Reason'] == False]
unique_missing_ids = missing_reasons['Batch_Id'].unique()

print(f"Total number of delayed batches: {delayed_batches['Batch_Id'].nunique()}")
print(f"Number of batches delayed without a recorded fault reason: {len(unique_missing_ids)}")
if len(unique_missing_ids) > 0:
  print(f"Hidden Missing Time Patch Numbers: {unique_missing_ids}")
else:
   print("All delayed patches have recorded fault reasons.")

Total number of delayed batches: 35
Number of batches delayed without a recorded fault reason: 0
All delayed patches have recorded fault reasons.


In [90]:
##	Are frequent minor failures more dangerous than a single major failure? 
failure_analysis = complete_table.groupby('Description')['Duration_Minutes'].agg(['count', 'mean', 'sum']).reset_index()
failure_analysis.columns = ['Description', 'Frequency', 'Avg_Minutes_Per_Incident', 'Total_Lost_Time']
print("Analysis of frequent versus large faults:")
print(failure_analysis.sort_values(by='Frequency', ascending=False).round(2))

Analysis of frequent versus large faults:
           Description  Frequency  Avg_Minutes_Per_Incident  Total_Lost_Time
7   Machine adjustment         12                     27.67            332.0
8      Machine failure         11                     23.09            254.0
4   Inventory shortage          9                     25.00            225.0
1   Batch coding error          6                     24.17            145.0
9                Other          6                     12.33             74.0
0         Batch change          5                     32.00            160.0
2    Calibration error          3                     16.33             49.0
5         Label switch          3                     11.00             33.0
10       Product spill          3                     19.00             57.0
6       Labeling error          2                     21.00             42.0
3    Conveyor belt jam          1                     17.00             17.0


                                                               Operator Analysis

In [97]:
##Which operator recorded the highest number of batches? 
operator_batches = complete_table.groupby('Operator_name')['Batch_Id'].nunique().sort_values(ascending=False)
print(operator_batches)
print(f"\nThe operator with the highest number of batches is: {operator_batches.idxmax()}")

Operator_name
Charlie    11
Dee        11
Dennis      8
Mac         8
Name: Batch_Id, dtype: int64

The operator with the highest number of batches is: Charlie


In [98]:
##Which operator has the lowest average production time for the same product? 
avg_production_time = complete_table.groupby(['Product_Id', 'Operator_name'])['Production_duration'].mean().reset_index()
fastest_operators = avg_production_time.loc[avg_production_time.groupby('Product_Id')['Production_duration'].idxmin()]
fastest_operators = fastest_operators.sort_values(by='Product_Id').reset_index(drop=True)
print("Workers with the lowest average (fastest) production time per product:")
fastest_operators

Workers with the lowest average (fastest) production time per product:


,Product_Id,Operator_name,Production_duration
0,CO-2L,Mac,130.000000
1,CO-600,Charlie,92.285714
2,DC-600,Dee,80.000000
3,LE-600,Charlie,73.500000
4,OR-600,Mac,135.000000
5,RB-600,Dennis,98.250000


In [100]:
##What is each operator’s ON TIME percentage? 
operator_status = complete_table.groupby(['Operator_name', 'Batch_Status'])['Batch_Id'].nunique().unstack(fill_value=0)
operator_status['Total_Batches'] = operator_status.sum(axis=1)
operator_status['On_Time_Percentage_%'] = (operator_status['On_time'] / operator_status['Total_Batches']) * 100
operator_status['On_Time_Percentage_%'] = operator_status['On_Time_Percentage_%'].round(2)
operator_status = operator_status.sort_values(by='On_Time_Percentage_%', ascending=False)
print("ON TIME ratio per factor:")
operator_status[['On_time', 'Total_Batches', 'On_Time_Percentage_%']]

ON TIME ratio per factor:


Batch_Status,On_time,Total_Batches,On_Time_Percentage_%
Operator_name,,,
Mac,1,8,12.50
Charlie,1,11,9.09
Dee,1,11,9.09
Dennis,0,8,0.00


In [101]:
# 1. حساب الفرق (التأخير) بين الوقت الفعلي والوقت المثالي لكل Batch
complete_table['Time_Deviation'] = complete_table['Production_duration'] - complete_table['Min batch time']
avg_deviation = complete_table.groupby('Operator_name')['Time_Deviation'].mean().reset_index()
avg_deviation = avg_deviation.sort_values(by='Time_Deviation', ascending=True).reset_index(drop=True)
print("Average difference between actual time and ideal time for each worker (in minutes):")
print(avg_deviation)
best_operator = avg_deviation.iloc[0]['Operator_name']
best_time = avg_deviation.iloc[0]['Time_Deviation']
print(f"\nThe factor closest to the ideal time is: {best_operator} with an average increase of {best_time:.2f} just a minute over the desired time.")

Average difference between actual time and ideal time for each worker (in minutes):
  Operator_name  Time_Deviation
0           Dee       41.350000
1           Mac       43.857143
2        Dennis       43.916667
3       Charlie       44.944444

The factor closest to the ideal time is: Dee with an average increase of 41.35 just a minute over the desired time.


In [102]:
##Which operator is associated with the highest number of operator errors? 
operator_errors_only = complete_table[complete_table['Operator Error'] == 'Yes']
errors_count = operator_errors_only['Operator_name'].value_counts()
print("Number of operating errors (Operator Errors) per operator:")
print(errors_count)
highest_errors_operator = errors_count.idxmax()
highest_errors_num = errors_count.max()

print(f"\n The factor associated with the largest number of operating errors is: {highest_errors_operator} with {highest_errors_num} errors.")

Number of operating errors (Operator Errors) per operator:
Operator_name
Dee        10
Charlie     9
Mac         7
Dennis      6
Name: count, dtype: int64

 The factor associated with the largest number of operating errors is: Dee with 10 errors.


In [103]:
####What type of human error is most common for each operator? 
human_errors = complete_table[complete_table['Operator Error'] == 'Yes']
error_counts = human_errors.groupby(['Operator_name', 'Description']).size().reset_index(name='Error_Count')
most_common_errors = error_counts.loc[error_counts.groupby('Operator_name')['Error_Count'].idxmax()]
most_common_errors = most_common_errors.sort_values(by='Error_Count', ascending=False).reset_index(drop=True)
most_common_errors.columns = ['Operator Name', 'Most Common Error', 'Number of Occurrences']
print("Most frequent human error per worker:")
most_common_errors

Most frequent human error per worker:


,Operator Name,Most Common Error,Number of Occurrences
0,Charlie,Machine adjustment,4
1,Dennis,Machine adjustment,4
2,Dee,Machine adjustment,3
3,Mac,Batch change,3


In [104]:
##•	How many downtime minutes did each operator contribute? 
operator_downtime = complete_table.groupby('Operator_name')['Duration_Minutes'].sum().reset_index()
operator_downtime = operator_downtime.sort_values(by='Duration_Minutes', ascending=False).reset_index(drop=True)
operator_downtime.columns = ['Operator Name', 'Total Downtime (Minutes)']
print("Total failure minutes per worker:")
operator_downtime

Total failure minutes per worker:


,Operator Name,Total Downtime (Minutes)
0,Charlie,384.0
1,Dee,370.0
2,Mac,332.0
3,Dennis,302.0


In [105]:
##•	Which operator restores operation fastest after a failure? 
machine_failures = complete_table[complete_table['Description'] == 'Machine failure']
repair_times = machine_failures.groupby('Operator_name')['Duration_Minutes'].mean().reset_index()
repair_times = repair_times.sort_values(by='Duration_Minutes', ascending=True).reset_index(drop=True)
repair_times.columns = ['Operator Name', 'Average Repair Time (Minutes)']
print("Average machine failure repair time (Machine failure) per worker:")
print(repair_times)
if not repair_times.empty:
    fastest_operator = repair_times.iloc[0]['Operator Name']
    fastest_time = repair_times.iloc[0]['Average Repair Time (Minutes)']
    print(f"\nThe fastest restart operator after a crash is: {fastest_operator} with average {fastest_time: .2f} minutes.")
else:
    print("\nNo machine failures recorded in the data.")

Average machine failure repair time (Machine failure) per worker:
  Operator Name  Average Repair Time (Minutes)
0           Dee                      18.000000
1        Dennis                      22.000000
2           Mac                      22.500000
3       Charlie                      28.333333

The fastest restart operator after a crash is: Dee with average  18.00 minutes.


In [106]:
##Does operator efficiency drop when moving from 600ml products to 2L products? 
if 'Time_Deviation' not in complete_table.columns:
    complete_table['Time_Deviation'] = complete_table['Production_duration'] - complete_table['Min batch time']
operator_size_efficiency = complete_table.groupby(['Operator_name', 'Size'])['Time_Deviation'].mean().unstack(fill_value=0)
size_stats = complete_table.groupby(['Size', 'Batch_Status'])['Batch_Id'].nunique().unstack(fill_value=0)
size_stats['Total_Batches'] = size_stats.sum(axis=1)
size_stats['Delay_Percentage_%'] = (size_stats['Delay'] / size_stats['Total_Batches']) * 100
print("1. Average delay minutes per worker by product size (600ml vs. 2000ml):")
print(operator_size_efficiency.round(2))
print("\n" + "="*50 + "\n")
print("2. General statistics: Total delay rate by product size:")
print(size_stats[['Total_Batches', 'Delay', 'Delay_Percentage_%']].round(2))

1. Average delay minutes per worker by product size (600ml vs. 2000ml):
Size           2000 ml  600 ml
Operator_name                 
Charlie          75.57   25.45
Dee               0.00   41.35
Dennis           54.00   41.90
Mac              32.00   45.83


2. General statistics: Total delay rate by product size:
Batch_Status  Total_Batches  Delay  Delay_Percentage_%
Size                                                  
2000 ml                   5      5              100.00
600 ml                   33     30               90.91


In [107]:
##Is operator performance consistent across shifts? 
if 'Time_Deviation' not in complete_table.columns:
    complete_table['Time_Deviation'] = complete_table['Production_duration'] - complete_table['Min batch time']
operator_shift_stats = complete_table.groupby(['Operator_name', 'shift', 'Batch_Status'])['Batch_Id'].nunique().unstack(fill_value=0)
operator_shift_stats['Total_Batches'] = operator_shift_stats.sum(axis=1)
operator_shift_stats['Delay_Percentage_%'] = (operator_shift_stats['Delay'] / operator_shift_stats['Total_Batches']) * 100
operator_shift_time = complete_table.groupby(['Operator_name', 'shift'])['Time_Deviation'].mean().unstack(fill_value=0)
print("1. Delay ratio per worker by shift (Shift):")
print(operator_shift_stats[['Total_Batches', 'Delay_Percentage_%']].round(2))
print("\n" + "="*50 + "\n")
print("2. Average delay minutes per worker by shift (Shift):")
print(operator_shift_time.round(2))

1. Delay ratio per worker by shift (Shift):
Batch_Status             Total_Batches  Delay_Percentage_%
Operator_name shift                                       
Charlie       Afternoon             10               90.00
              Night                  1              100.00
Dee           Morning                6               83.33
              Night                  5              100.00
Dennis        Afternoon              1              100.00
              Morning                7              100.00
Mac           Afternoon              5               80.00
              Morning                2              100.00
              Night                  1              100.00


2. Average delay minutes per worker by shift (Shift):
shift          Afternoon  Morning  Night
Operator_name                           
Charlie            46.71     0.00  15.00
Dee                 0.00    31.22  49.64
Dennis             40.00    44.70   0.00
Mac                37.50    62.50  32.00


In [131]:
###∑Min_Batch_Time∑Actual Duration×100\frac{\sum Min\_Batch\_Time}{\sum Actual\ Duration} \times 100∑Actual Duration∑Min_Batch_Time×100 
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id'])
total_min_time = unique_batches['Min batch time'].sum()
total_actual_time = unique_batches['Production_duration'].sum()

overall_efficiency = (total_min_time / total_actual_time) * 100

print(f"Total Perfect Time (Min Batch Time): {total_min_time} minutes")
print(f"Total Actual Time (Actual Duration): {total_actual_time} minutes")
print(f"Overall Efficiency (Overall Efficiency): {overall_efficiency: .2f}%\n")

print("="*50 + "\n")
operator_efficiency = unique_batches.groupby('Operator_name').agg({
    'Min batch time': 'sum',
    'Production_duration': 'sum'
}).reset_index()

operator_efficiency['Efficiency_%'] = (operator_efficiency['Min batch time'] / operator_efficiency['Production_duration']) * 100

operator_efficiency = operator_efficiency.sort_values(by='Efficiency_%', ascending=False).reset_index(drop=True)

operator_efficiency['Efficiency_%'] = operator_efficiency['Efficiency_%'].round(2)

print("Production efficiency per worker (Operator Efficiency):")
print(operator_efficiency)

Total Perfect Time (Min Batch Time): 2470 minutes
Total Actual Time (Actual Duration): 3858 minutes
Overall Efficiency (Overall Efficiency):  64.02%


Production efficiency per worker (Operator Efficiency):
  Operator_name  Min batch time  Production_duration  Efficiency_%
0       Charlie             774                 1158         66.84
1           Dee             660                 1030         64.08
2        Dennis             518                  820         63.17
3           Mac             518                  850         60.94


In [109]:
##•	Average production time without Operator Errors. 
batches_with_errors = complete_table[complete_table['Operator Error'] == 'Yes']['Batch_Id'].unique()
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id'])
error_free_batches = unique_batches[~unique_batches['Batch_Id'].isin(batches_with_errors)]
avg_time_no_errors = error_free_batches['Production_duration'].mean()
avg_time_all = unique_batches['Production_duration'].mean()
print(f"Number of Batches without human errors: {len(error_free_batches)} patch")
print(f"Average production time (no human errors): {avg_time_no_errors: .2f} minutes")
print(f"Average overall production time (including errors): {avg_time_all:.2f} minutes")

time_saved = avg_time_all - avg_time_no_errors
print(f"\n=> Human errors increase average production time by about {time_saved:.2f} minutes per patch!")


Number of Batches without human errors: 13 patch
Average production time (no human errors):  83.46 minutes
Average overall production time (including errors): 101.53 minutes

=> Human errors increase average production time by about 18.06 minutes per patch!


In [110]:
##Estimated waste or lost time caused by each operator. 
operator_waste = complete_table[complete_table['Operator Error'] == 'Yes']
lost_time_by_operator = operator_waste.groupby('Operator_name')['Duration_Minutes'].sum().reset_index()
lost_time_by_operator = lost_time_by_operator.sort_values(by='Duration_Minutes', ascending=False).reset_index(drop=True)
lost_time_by_operator.columns = ['Operator Name', 'Estimated Waste/Lost Time (Minutes)']

print("Total time wasted (in minutes) due to errors by each worker:")
lost_time_by_operator

Total time wasted (in minutes) due to errors by each worker:


,Operator Name,Estimated Waste/Lost Time (Minutes)
0,Charlie,228.0
1,Dee,192.0
2,Mac,192.0
3,Dennis,164.0


In [95]:
##Does continuous working time affect operator efficiency? 
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()

unique_batches = unique_batches.sort_values(by=['Date', 'Operator_name', 'Start Time'])
unique_batches['Work_Minutes_So_Far'] = unique_batches.groupby(['Date', 'Operator_name'])['Production_duration'].cumsum() - unique_batches['Production_duration']

bins = [-1, 120, 240, 360]
labels = ['0-2 Hours', '2-4 Hours', '4-6 Hours']
unique_batches['Work_Window'] = pd.cut(unique_batches['Work_Minutes_So_Far'], bins=bins, labels=labels)

fatigue_impact = unique_batches.groupby('Work_Window')['Time_Deviation'].mean().reset_index()
fatigue_impact.columns = ['Continuous working hours', 'Average delay (minutes)']

print("Analysis of the impact of continuous working hours on operator efficiency:")
display(fatigue_impact.round(2))

Analysis of the impact of continuous working hours on operator efficiency:


C:\Users\Midoz\AppData\Local\Temp\ipykernel_10444\948618125.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fatigue_impact = unique_batches.groupby('Work_Window')['Time_Deviation'].mean().reset_index()


,Continuous working hours,Average delay (minutes)
0,0-2 Hours,32.55
1,2-4 Hours,33.27
2,4-6 Hours,59.33


                                                         Shift Analysis

In [116]:
##	Which shift produces the highest batch volume? 
shift_volume = complete_table.groupby('shift')['Batch_Id'].count().reset_index()
shift_volume = shift_volume.sort_values(by='Batch_Id', ascending=False).reset_index(drop=True)
shift_volume.columns = ['Shift', 'Total Batch Volume']
print("Production volume (number of Batches) per shift:")
print(shift_volume)
top_shift = shift_volume.iloc[0]['Shift']
top_volume = shift_volume.iloc[0]['Total Batch Volume']
print(f"\nThe highest shift in terms of production volume is: {top_shift} with {top_volume} patch.")

Production volume (number of Batches) per shift:
       Shift  Total Batch Volume
0  Afternoon                  27
1    Morning                  23
2      Night                  14

The highest shift in terms of production volume is: Afternoon with 27 patch.


In [117]:
##•	Does a certain shift have more downtime? 
shift_downtime = complete_table.groupby('shift')['Duration_Minutes'].sum().reset_index()
shift_downtime = shift_downtime.sort_values(by='Duration_Minutes', ascending=False).reset_index(drop=True)
shift_downtime.columns = ['Shift', 'Total Downtime (Minutes)']
print("Total crash time (Downtime) per shift:")
print(shift_downtime)
top_downtime_shift = shift_downtime.iloc[0]['Shift']
top_downtime_mins = shift_downtime.iloc[0]['Total Downtime (Minutes)']
print(f"\nThe shift with the longest crash time is: {top_downtime_shift} for a total of {top_downtime_mins} minutes.")

Total crash time (Downtime) per shift:
       Shift  Total Downtime (Minutes)
0  Afternoon                     584.0
1    Morning                     534.0
2      Night                     270.0

The shift with the longest crash time is: Afternoon for a total of 584.0 minutes.


In [118]:
##Are these failures machine-related or material shortages? 
target_failures = complete_table[complete_table['Description'].isin(['Machine failure', 'Inventory shortage'])]
failure_comparison = target_failures.groupby('Description').agg(
    Occurrences=('Batch_Id', 'nunique'),
    Total_Downtime_Minutes=('Duration_Minutes', 'sum')
).reset_index()
total_mins = failure_comparison['Total_Downtime_Minutes'].sum()
failure_comparison['Percentage_of_Time_%'] = (failure_comparison['Total_Downtime_Minutes'] / total_mins) * 100
failure_comparison['Percentage_of_Time_%'] = failure_comparison['Percentage_of_Time_%'].round(2)
print("Comparison between machine failures (Machine failure) and material shortages (Inventory shortage):")
failure_comparison

Comparison between machine failures (Machine failure) and material shortages (Inventory shortage):


,Description,Occurrences,Total_Downtime_Minutes,Percentage_of_Time_%
0,Inventory shortage,9,225.0,46.97
1,Machine failure,11,254.0,53.03


In [119]:
##•	Which shift performs best against target time? 
if 'Time_Deviation' not in complete_table.columns:
    complete_table['Time_Deviation'] = complete_table['Production_duration'] - complete_table['Min batch time']

shift_performance = complete_table.groupby('shift')['Time_Deviation'].mean().reset_index()
shift_performance = shift_performance.sort_values(by='Time_Deviation', ascending=True).reset_index(drop=True)
shift_performance.columns = ['Shift', 'Average Delay vs Target (Minutes)']

print("Average delay from target time per shift (less is better):")
print(shift_performance.round(2))
best_shift = shift_performance.iloc[0]['Shift']
best_delay = shift_performance.iloc[0]['Average Delay vs Target (Minutes)']

print("\n" + "="*60)
print(f"The best performing shift compared to the target time is: {best_shift} with an average delay of {best_delay: .2f} minutes for Patch.")
print("="*60)

Average delay from target time per shift (less is better):
       Shift  Average Delay vs Target (Minutes)
0    Morning                              42.52
1  Afternoon                              43.48
2      Night                              44.64

The best performing shift compared to the target time is: Morning with an average delay of  42.52 minutes for Patch.


In [120]:
##Do batches spanning two shifts take longer due to handover issues?
import pandas as pd
def get_shift_from_time(t):
    
    if isinstance(t, str):
        t = pd.to_datetime(t).time()
    
    hour = t.hour
    if 6 <= hour < 14:
        return 'Morning'
    elif 14 <= hour < 22:
        return 'Afternoon'
    else: 
        return 'Night'

unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()
if 'Time_Deviation' not in unique_batches.columns:
    unique_batches['Time_Deviation'] = unique_batches['Production_duration'] - unique_batches['Min batch time']
unique_batches['End_Shift'] = unique_batches['End Time'].apply(get_shift_from_time)
unique_batches['Handover_Issue'] = unique_batches['shift'] != unique_batches['End_Shift']
handover_impact = unique_batches.groupby('Handover_Issue')['Time_Deviation'].agg(['count', 'mean']).reset_index()

handover_impact['Handover_Issue'] = handover_impact['Handover_Issue'].replace({
    True: 'Yes (Spans 2 Shifts)', 
    False: 'No (Same Shift)'
})
handover_impact.columns = ['Spans Handover?', 'Total Batches', 'Average Delay (Minutes)']

print("Effect of shift delivery time (Handover) on average delay:")
print(handover_impact.round(2))

same_shift_delay = handover_impact[handover_impact['Spans Handover?'] == 'No (Same Shift)']['Average Delay (Minutes)'].values
cross_shift_delay = handover_impact[handover_impact['Spans Handover?'] == 'Yes (Spans 2 Shifts)']['Average Delay (Minutes)'].values

if len(same_shift_delay) > 0 and len(cross_shift_delay) > 0:
    diff = cross_shift_delay[0] - same_shift_delay[0]
    if diff > 0:
        print(f"\n Yes, matches that pass the shift delivery time are delayed more by an average of {diff:.2f} additional minutes compared to those that end on the same shift!")
    else:
        print(f"\n No, delivery does not appear to cause additional delay (difference: {diff:.2f} min).")

Effect of shift delivery time (Handover) on average delay:
        Spans Handover?  Total Batches  Average Delay (Minutes)
0       No (Same Shift)             29                    28.93
1  Yes (Spans 2 Shifts)              9                    61.00

 Yes, matches that pass the shift delivery time are delayed more by an average of 32.07 additional minutes compared to those that end on the same shift!


In [60]:
##Is any shift associated with more Operator Errors? 
operator_errors_only = complete_table[complete_table['Operator Error'] == 'Yes']
shift_errors = operator_errors_only['shift'].value_counts().reset_index()
shift_errors.columns = ['Shift', 'Operator Errors Count']
print("Number of operating errors (Operator Errors) per shift:")

print(shift_errors)

if not shift_errors.empty:
    top_shift = shift_errors.iloc[0]['Shift']
    top_errors = shift_errors.iloc[0]['Operator Errors Count']
    print(f"\n=> The pinkest human errors are: {top_shift} with a total of {top_errors} errors.")
else:
    print("\n No human errors recorded in the data.")

عدد أخطاء التشغيل (Operator Errors) لكل وردية:
       Shift  Operator Errors Count
0  Afternoon                     15
1    Morning                      9
2      Night                      8

=> أكثر وردية بيحصل فيها أخطاء بشرية هي: Afternoon بإجمالي 15 أخطاء.


                                                             Product Analysis

In [121]:
##Do 2L products consistently have higher delays than 600ml products? 
if 'Time_Deviation' not in complete_table.columns:
    complete_table['Time_Deviation'] = complete_table['Production_duration'] - complete_table['Min batch time']
size_delay_avg = complete_table.groupby('Size')['Time_Deviation'].mean().reset_index()
size_delay_avg.columns = ['Size', 'Average Delay (Minutes)']
size_status_counts = complete_table.groupby(['Size', 'Batch_Status'])['Batch_Id'].nunique().unstack(fill_value=0)
size_status_counts['Total_Batches'] = size_status_counts.sum(axis=1)
if 'Delay' in size_status_counts.columns:
    size_status_counts['Delay_Percentage_%'] = (size_status_counts['Delay'] / size_status_counts['Total_Batches']) * 100
else:
    size_status_counts['Delay_Percentage_%'] = 0
size_comparison = pd.merge(size_delay_avg, size_status_counts[['Delay_Percentage_%']].reset_index(), on='Size')

print("Comparison of delay between 600ml and 2000ml (2L) products:")
print(size_comparison.round(2))


avg_600 = size_comparison[size_comparison['Size'] == '600 ml']['Average Delay (Minutes)'].values
avg_2000 = size_comparison[size_comparison['Size'] == '2000 ml']['Average Delay (Minutes)'].values

if len(avg_600) > 0 and len(avg_2000) > 0:
    if avg_2000[0] > avg_600[0]:
        print(f"\n Yes, 2L products are further delayed. The average delay of 2L is {avg_2000[0]:.2f} minutes, compared to {avg_600[0]:.2f} minutes for 600ml.")
    else:
        print(f"\n No, 600ml products are delayed more or equally.")

Comparison of delay between 600ml and 2000ml (2L) products:
      Size  Average Delay (Minutes)  Delay_Percentage_%
0  2000 ml                    63.73              100.00
1   600 ml                    39.17               90.91

=> Yes, 2L products are further delayed. The average delay of 2L is 63.73 minutes, compared to 39.17 minutes for 600ml.


In [122]:
##Does a specific flavor (e.g. Root Berry or Lemon Lime) cause more machine issues or Product Spill incidents? 
target_issues = ['Machine failure', 'Machine adjustment', 'Product spill']
flavor_issues = complete_table[complete_table['Description'].isin(target_issues)]
flavor_breakdown = flavor_issues.groupby(['Flavor', 'Description'])['Batch_Id'].nunique().unstack(fill_value=0)
flavor_breakdown['Total Specific Issues'] = flavor_breakdown.sum(axis=1)
flavor_breakdown = flavor_breakdown.sort_values(by='Total Specific Issues', ascending=False)

print("Number of machine problems and product spills (Product Spill) per flavor:")
print(flavor_breakdown)

if not flavor_breakdown.empty:
    top_flavor = flavor_breakdown.index[0]
    top_issues = flavor_breakdown.iloc[0]['Total Specific Issues']
    print(f"\n=> The flavor that causes the most machine problems and spills is: {top_flavor} with a total of {top_issues} incidents.")
else:
    print("\n There are no recorded incidents of product spills or machine malfunctions in the current data.")

Number of machine problems and product spills (Product Spill) per flavor:
Description  Machine adjustment  Machine failure  Product spill  \
Flavor                                                            
Cola                          6                7              3   
Root Berry                    4                1              0   
Diet Cola                     0                2              0   
Lemon lime                    2                0              0   
Orange                        0                1              0   

Description  Total Specific Issues  
Flavor                              
Cola                            16  
Root Berry                       5  
Diet Cola                        2  
Lemon lime                       2  
Orange                           1  

=> The flavor that causes the most machine problems and spills is: Cola with a total of 16 incidents.


In [123]:
##How many minutes are lost during product changeovers (Factor ID:2)? 
changeover_lost_time = complete_table[complete_table['Factor_Id'] == 2]['Duration_Minutes'].sum()

print(f"Total time wasted due to changing products (Product Changeovers): {changeover_lost_time} minutes.")

Total time wasted due to changing products (Product Changeovers): 160.0 minutes.


In [124]:
##Does this vary by product type? 
changeover_data = complete_table[complete_table['Factor_Id'] == 2]
product_changeover = changeover_data.groupby('Product_Id')['Duration_Minutes'].agg(['count', 'mean', 'sum']).reset_index()
product_changeover.columns = ['Product_Id', 'Changeover_Count', 'Average_Minutes', 'Total_Minutes']
size_changeover = changeover_data.groupby('Size')['Duration_Minutes'].agg(['count', 'mean', 'sum']).reset_index()
size_changeover.columns = ['Size', 'Changeover_Count', 'Average_Minutes', 'Total_Minutes']

print("1. Analyze product change time by product type:")
print(product_changeover.sort_values(by='Average_Minutes', ascending=False).round(2))

print("\n" + "="*50 + "\n")

print("2. Analyze product change time by product size (Size):")
print(size_changeover.sort_values(by='Average_Minutes', ascending=False).round(2))

1. Analyze product change time by product type:
  Product_Id  Changeover_Count  Average_Minutes  Total_Minutes
2     OR-600                 1            60.00           60.0
1     LE-600                 3            26.67           80.0
0     CO-600                 1            20.00           20.0


2. Analyze product change time by product size (Size):
     Size  Changeover_Count  Average_Minutes  Total_Minutes
0  600 ml                 5             32.0          160.0


In [125]:
##Is any product more associated with Labeling Errors or Label Switch failures? 
labeling_errors = complete_table[complete_table['Factor_Id'] == 3]
product_label_issues = labeling_errors.groupby('Product_Id').size().reset_index(name='Error_Count')
total_label_errors = product_label_issues['Error_Count'].sum()
product_label_issues['Contribution_%'] = (product_label_issues['Error_Count'] / total_label_errors) * 100
print("Analysis of label errors (Labeling errors) by product type:")
print(product_label_issues.sort_values(by='Error_Count', ascending=False).reset_index(drop=True))

Analysis of label errors (Labeling errors) by product type:
  Product_Id  Error_Count  Contribution_%
0      CO-2L            1            50.0
1     RB-600            1            50.0


In [126]:
##Based on data, which product is the easiest (usually On-Time) and which is the most difficult for the line? 
product_performance = complete_table.groupby(['Product_Id', 'Batch_Status'])['Batch_Id'].nunique().unstack(fill_value=0)
product_performance['Total'] = product_performance.sum(axis=1)
product_performance['On_Time_Rate_%'] = (product_performance['On_time'] / product_performance['Total']) * 100
product_delay_avg = complete_table.groupby('Product_Id')['Production_duration'].mean() - complete_table.groupby('Product_Id')['Min batch time'].mean()
product_performance['Avg_Delay_Mins'] = product_delay_avg
print("Analysis of production efficiency by product (easier than harder):")
print(product_performance[['On_time', 'Total', 'On_Time_Rate_%', 'Avg_Delay_Mins']].sort_values(by='On_Time_Rate_%', ascending=False).round(2))

Analysis of production efficiency by product (easier than harder):
Batch_Status  On_time  Total  On_Time_Rate_%  Avg_Delay_Mins
Product_Id                                                  
DC-600              1      4           25.00           35.00
LE-600              1      6           16.67           29.33
CO-600              1     15            6.67           39.72
CO-2L               0      5            0.00           63.73
OR-600              0      1            0.00           75.00
RB-600              0      7            0.00           41.73


In [127]:
##A matrix showing average production time for each product in each shift. 
production_matrix = df_productivity.pivot_table(
    values='Production_duration', 
    index='Product_Id', 
    columns='shift', 
    aggfunc='mean'
)
ordered_shifts = ['Morning', 'Afternoon', 'Night']
production_matrix = production_matrix.reindex(columns=[s for s in ordered_shifts if s in production_matrix.columns])

print("Matrix of average production time (in minutes) for each product by shift:")
production_matrix.round(2)

Matrix of average production time (in minutes) for each product by shift:


shift,Morning,Afternoon,Night
Product_Id,,,
CO-2L,152.00,161.67,130.00
CO-600,90.00,95.80,97.50
DC-600,95.00,82.50,NaN
LE-600,NaN,88.17,NaN
OR-600,135.00,NaN,NaN
RB-600,91.67,NaN,100.75


In [128]:
##Average downtime minutes for each product. 
batch_total_downtime = complete_table.groupby('Batch_Id')['Duration_Minutes'].sum().reset_index()
product_downtime_mapping = pd.merge(
    df_productivity[['Batch_Id', 'Product_Id']], 
    batch_total_downtime, 
    on='Batch_Id', 
    how='left'
).fillna(0) 
avg_downtime_per_product = product_downtime_mapping.groupby('Product_Id')['Duration_Minutes'].mean().reset_index()
avg_downtime_per_product.columns = ['Product_Id', 'Average Downtime (Minutes)']
print("Average crash minutes per product (per patch):")
print(avg_downtime_per_product.round(2).sort_values(by='Average Downtime (Minutes)', ascending=False).reset_index(drop=True))

Average crash minutes per product (per patch):
  Product_Id  Average Downtime (Minutes)
0     OR-600                       75.00
1      CO-2L                       55.40
2     RB-600                       36.86
3     CO-600                       32.93
4     DC-600                       28.75
5     LE-600                       28.17


In [129]:
##Line utilization ratio by Small vs Large products. 
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()
def categorize_size(size):
    if '600' in str(size):
        return 'Small (600ml)'
    else:
        return 'Large (2000ml)'

unique_batches['Category'] = unique_batches['Size'].apply(categorize_size)
utilization_stats = unique_batches.groupby('Category').agg({
    'Min batch time': 'sum',
    'Production_duration': 'sum'
}).reset_index()
utilization_stats['Utilization_Ratio_%'] = (utilization_stats['Min batch time'] / utilization_stats['Production_duration']) * 100

print("Line Utilization Ratio (Line Utilization Ratio) by Product Size:")
print(utilization_stats[['Category', 'Min batch time', 'Production_duration', 'Utilization_Ratio_%']].round(2))

Line Utilization Ratio (Line Utilization Ratio) by Product Size:
         Category  Min batch time  Production_duration  Utilization_Ratio_%
0  Large (2000ml)             490                  767                63.89
1   Small (600ml)            1980                 3091                64.06


In [130]:
##Is the 38-minute difference actually respected, or do 2L products consume much more time in reality? 
unique_batches = complete_table.drop_duplicates(subset=['Batch_Id']).copy()
actual_avg_time = unique_batches.groupby('Size')['Production_duration'].mean()
avg_time_600 = actual_avg_time.get('600 ml', 0)
avg_time_2000 = actual_avg_time.get('2000 ml', 0)
target_diff = 98 - 60
actual_diff = avg_time_2000 - avg_time_600
extra_lost_time = actual_diff - target_diff

print(f"Target Difference (Target Difference): {target_diff} minutes")
print(f"Actual Difference Actually (Actual Average Difference): {actual_diff: .2f} minutes")
print(f"Overtime lost in the above-planned 2L: {extra_lost_time: .2f} minutes")

print("\n" + "="*50)
if actual_diff > target_diff:
    print(f"Conclusion: 2L consumes {extra_lost_time: .2f} minutes more 'actual' time than expected difference!")
else:
    print("Conclusion: The actual difference is close to the target difference.")

Target Difference (Target Difference): 38 minutes
Actual Difference Actually (Actual Average Difference):  59.73 minutes
Overtime lost in the above-planned 2L:  21.73 minutes

Conclusion: 2L consumes  21.73 minutes more 'actual' time than expected difference!
